# Phase 1 -- Notebook p1_07: Model C trainer (prefix-LM, resumable)

**Model C** reads the prompt in both directions (every prompt token sees
the whole prompt) and writes the continuation left to right -- the same
shape as the test task. Half of its training windows are cut exactly like
the test (long prompt -> continuation, loss only on the continuation); the
other half are plain next-word prediction. Tokenizer **T2** (SentencePiece).
~113M parameters.

**How to run it**
1. Inputs: `legallm-phase1-data`, `legallm-tokenizers`. (For a resumed
   session also `legallm-checkpoints-c`.)
2. Secrets on: `KAGGLE_API_TOKEN`. GPU T4 x2, internet on.
3. `SMOKE_MODE = False` is already set -> **Save Version -> Save & Run All**,
   then stop the editor session.
4. Watch the log for the first ~15 min: `Self-test passed`, `DataParallel
   self-test passed`, `Schedule:`, and `step 0 loss=` around 10.4.

**Time:** ~8.5h training + ~20 min setup and scoring.


## 1. Imports, GPUs, and input folders

In [1]:
import datetime
import gc
import itertools
import json
import math
import random
import shutil
import subprocess
import time
import zipfile
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

print("PyTorch version:", torch.__version__)
N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} "
          f"({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected. Set Accelerator -> GPU T4 x2.")

KAGGLE_INPUT = Path("/kaggle/input")


def find_input_dir(marker, dataset_label):
    hits = sorted(KAGGLE_INPUT.rglob(marker)) if KAGGLE_INPUT.exists() else []
    if not hits:
        attached = sorted(p.name for p in KAGGLE_INPUT.glob("*")) if KAGGLE_INPUT.exists() else []
        raise FileNotFoundError(f"Could not find '{marker}' under /kaggle/input. Attach '{dataset_label}'. "
                                f"Attached now: {attached}")
    return hits[0].parent


INPUT_DATA_DIR = find_input_dir("dev_prompts.json", "legallm-phase1-data")
INPUT_TOKENIZER_DIR = find_input_dir("t2_unigram.model", "legallm-tokenizers")
OUTPUT_DIR = Path("/kaggle/working/phase1_model_c")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR      # flat on purpose (a sub-folder is uploaded as a zip)
print(f"Data: {INPUT_DATA_DIR}\nTokenizer: {INPUT_TOKENIZER_DIR}\nOutput: {OUTPUT_DIR}")


PyTorch version: 2.10.0+cu128
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)
Data: /kaggle/input/datasets/jbhusal/legallm-phase1-data
Tokenizer: /kaggle/input/datasets/jbhusal/legallm-tokenizers
Output: /kaggle/working/phase1_model_c


## 2. Run mode & config

`TIME_BUDGET_MIN` = this session's training time; `TOTAL_TRAIN_HOURS` =
the whole C run (the learning-rate schedule is sized to it and fully winds
down by the end). Defaults: one 8.5h session.


In [2]:
SMOKE_MODE = False  # True = 15-min pipeline check (optional)

MODEL_C_CONFIG = dict(
    vocab_size=32_000,        # T2 (SentencePiece Unigram)
    d_model=768,
    n_layer=14,
    n_head=12,
    n_kv_head=4,              # GQA 3:1
    d_head=64,
    d_ff=3072,                # GELU MLP, 4x
    max_seq_len=2048,
    qk_norm=True,
    bias=True,                # biases kept (BitFit in later phases)
    mlp="gelu",
    tokenizer_id="T2",
    attn_type="prefix",
    tied_embeddings=True,
)
ARCH_ID = "C"
CKPT_PREFIX = "Csmoke" if SMOKE_MODE else ARCH_ID

TRAIN_SEQ_LEN = 1024          # a full test example (~730 prompt + ~183 continuation T2 tokens) fits
TASK_FRACTION = 0.5           # share of windows cut like the test (prefix -> continuation)
PREFIX_MIN, PREFIX_MAX = 480, TRAIN_SEQ_LEN - 192
USE_BOTH_GPUS = True
TOKENS_PER_STEP_TARGET = 32_768

PEAK_LR = 6e-4                # B used 6e-4 and trained cleanly
MIN_LR = PEAK_LR / 10
WEIGHT_DECAY = 0.1
GRAD_CLIP_NORM = 1.0
Z_LOSS_COEF = 1e-4
LR_DECAY_STEPS_OVERRIDE = None

if SMOKE_MODE:
    TIME_BUDGET_MIN, TOTAL_TRAIN_HOURS, CHECKPOINT_EVERY_MIN, PUSH_DURING_TRAINING = 15, 0.25, 6, False
else:
    TIME_BUDGET_MIN = 510         # 8.5h this session
    TOTAL_TRAIN_HOURS = 8.5       # the whole C run
    CHECKPOINT_EVERY_MIN = 45
    PUSH_DURING_TRAINING = True

assert TIME_BUDGET_MIN <= 11 * 60 and TOTAL_TRAIN_HOURS * 60 >= TIME_BUDGET_MIN
print(f"SMOKE_MODE={SMOKE_MODE}  TIME_BUDGET_MIN={TIME_BUDGET_MIN}  TOTAL_TRAIN_HOURS={TOTAL_TRAIN_HOURS}")
print(json.dumps(MODEL_C_CONFIG, indent=2))


SMOKE_MODE=False  TIME_BUDGET_MIN=510  TOTAL_TRAIN_HOURS=8.5
{
  "vocab_size": 32000,
  "d_model": 768,
  "n_layer": 14,
  "n_head": 12,
  "n_kv_head": 4,
  "d_head": 64,
  "d_ff": 3072,
  "max_seq_len": 2048,
  "qk_norm": true,
  "bias": true,
  "mlp": "gelu",
  "tokenizer_id": "T2",
  "attn_type": "prefix",
  "tied_embeddings": true
}


## 3. Tokenizer (T2)

In [3]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load(str(INPUT_TOKENIZER_DIR / "t2_unigram.model"))
EOS_ID = sp.eos_id()
BANNED_IN_OUTPUT = sorted({sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()} - {-1})
assert EOS_ID >= 0 and sp.get_piece_size() <= MODEL_C_CONFIG["vocab_size"]


def encode(text):
    return sp.encode(text, out_type=int)


def decode(ids):
    return sp.decode(ids)


print(f"T2 vocab {sp.get_piece_size()}, <eos>={EOS_ID}, never generated: {BANNED_IN_OUTPUT}")
print("Round trip:", decode(encode("Section 302 of the Indian Penal Code, 1860.")))


T2 vocab 32000, <eos>=3, never generated: [0, 1, 2, 3]
Round trip: Section 302 of the Indian Penal Code, 1860.


## 4. The model

| Piece | A | B | **C** |
|---|---|---|---|
| Attention | causal | local window + global | **two-way over the prompt, causal over the continuation** |
| Training target | every next token | every next token | **continuation only (task windows) + every token (LM windows)** |
| MLP / biases | SwiGLU, none | SwiGLU, none | **GELU, biases kept** |
| Tokenizer | T1 | T1 | **T2** |

**The prefix mask:** key `k` is visible to query `q` if `k <= q` (causal)
**or** `k` is inside the prompt. So every prompt token sees the whole
prompt, and continuation tokens see the whole prompt plus the continuation
so far. The prompt never sees the continuation, so there's no leak -- the
self-test checks this, plus cached generation vs a full forward pass.


In [4]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(seq_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x [B,H,T,d]; cos/sin [B or 1, T, d/2]. Rotation in fp32."""
    x1, x2 = x[..., 0::2].float(), x[..., 1::2].float()
    cos, sin = cos[:, None].float(), sin[:, None].float()
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2).to(x.dtype)


def prefix_lm_mask(prefix_len, T, device):
    """Bool [B, 1, T, T]: key k visible to query q iff k <= q or k < prefix_len[b]."""
    idx = torch.arange(T, device=device)
    causal = idx[None, :, None] >= idx[None, None, :]
    in_prefix = idx[None, None, :] < prefix_len[:, None, None]
    return (causal | in_prefix)[:, None]


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh, b = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"], cfg["bias"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=b)
        self.k_proj = nn.Linear(d, kv * dh, bias=b)
        self.v_proj = nn.Linear(d, kv * dh, bias=b)
        self.o_proj = nn.Linear(h * dh, d, bias=b)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, attn_mask, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        n_rep = self.n_head // self.n_kv_head
        if n_rep > 1:
            k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
        if attn_mask is None:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class GeluMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.up_proj = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["bias"])
        self.down_proj = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["bias"])

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = Attention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = GeluMLP(cfg)

    def _inner(self, x, cos, sin, attn_mask, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, attn_mask, cache)
        return x + self.mlp(self.mlp_norm(x))

    def forward(self, x, cos, sin, attn_mask, cache=None, use_checkpoint=False):
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._inner, x, cos, sin, attn_mask, None, use_reentrant=False)
        return self._inner(x, cos, sin, attn_mask, cache)


class KVCache:
    def __init__(self, batch_size, n_kv_head, max_len, d_head, device, dtype):
        self.k = torch.zeros(batch_size, n_kv_head, max_len, d_head, device=device, dtype=dtype)
        self.v = torch.zeros_like(self.k)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class PrefixLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([Block(config) for _ in range(config["n_layer"])])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight
        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def encode(self, input_ids, attention_mask=None, mode="prefix", prefix_len=None):
        """Contract method: hidden states. mode 'causal' (prefix 0), 'prefix'
        (needs prefix_len [B]) or 'bidirectional' (whole input is prefix)."""
        B, T = input_ids.shape
        if mode == "causal" or prefix_len is None and mode == "prefix":
            prefix_len = torch.zeros(B, dtype=torch.long, device=input_ids.device)
        elif mode == "bidirectional":
            prefix_len = torch.full((B,), T, dtype=torch.long, device=input_ids.device)
        mask = prefix_lm_mask(prefix_len, T, input_ids.device) if bool((prefix_len > 0).any()) else None
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[None, :T], self.rope_sin[None, :T]
        use_ckpt = self.training and USE_GRAD_CHECKPOINTING
        for blk in self.blocks:
            x = blk(x, cos, sin, mask, use_checkpoint=use_ckpt)
        return self.norm_f(x)

    def forward(self, ids, prefix_len=None, targets=None):
        """No targets -> logits. With targets -> [1, 3] = [loss sum, number of
        loss tokens, sum of logsumexp^2 (z-loss)] over positions that predict
        continuation tokens (t >= prefix_len - 1). Sums (not means) so
        DataParallel replicas combine exactly."""
        use_amp = self.training and ids.is_cuda
        if prefix_len is None:
            prefix_len = torch.zeros(ids.shape[0], dtype=torch.long, device=ids.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = self.head(self.encode(ids, mode="prefix", prefix_len=prefix_len))
        if targets is None:
            return logits
        logits32 = logits.float()
        lse = torch.logsumexp(logits32, dim=-1)
        nll = lse - logits32.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        pos = torch.arange(ids.shape[1], device=ids.device)[None, :]
        w = (pos >= (prefix_len[:, None] - 1)).float()
        return torch.stack([(nll * w).sum(), w.sum(), (lse.pow(2) * w).sum()]).unsqueeze(0)

    def step(self, ids, positions, caches, attn_mask):
        """Generation step with a KV cache. ids/positions [B, T]; attn_mask
        bool [B, 1, T, cache_len + T]."""
        x = self.embed(ids)
        cos, sin = self.rope_cos[positions], self.rope_sin[positions]
        for blk, cache in zip(self.blocks, caches):
            x = blk(x, cos, sin, attn_mask, cache=cache)
        return self.head(self.norm_f(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (greedy, one prompt): the prompt is the prefix."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


@torch.no_grad()
def generate_batch(model, prompt_id_lists, max_new_tokens=288, max_context=None, rep_penalty=1.0,
                   no_repeat_ngram=0, target_words=None, banned=(), vocab_limit=None):
    """Greedy continuations for several prompts at once (left-padded). The
    prompt is the bidirectional prefix; continuation is causal. Optional:
    repetition penalty over generated tokens, n-gram blocking inside the
    continuation, stop once every row has > target_words words."""
    model.eval()
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    cfg = model.config
    max_context = max_context or cfg["max_seq_len"]
    prompts = [p[-(max_context - max_new_tokens):] for p in prompt_id_lists]
    B, T0 = len(prompts), max(len(p) for p in prompts)
    pad = torch.tensor([T0 - len(p) for p in prompts], device=device)
    ids = torch.zeros(B, T0, dtype=torch.long, device=device)
    for b, p in enumerate(prompts):
        ids[b, T0 - len(p):] = torch.tensor(p, device=device)
    total = T0 + max_new_tokens
    caches = [KVCache(B, cfg["n_kv_head"], total, cfg["d_head"], device, dtype) for _ in range(cfg["n_layer"])]
    idx = torch.arange(total, device=device)
    key_ok = idx[None, :] >= pad[:, None]
    positions = (idx[None, :] - pad[:, None]).clamp(min=0)
    # Prefill: the prompt attends to itself in both directions (no padding keys);
    # every query may see itself so padded rows are never fully masked.
    mask = (key_ok[:, None, :T0] | torch.eye(T0, dtype=torch.bool, device=device)[None])[:, None]
    logits = model.step(ids, positions[:, :T0], caches, mask)[:, -1, :].float()
    generated = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    for t in range(max_new_tokens):
        if banned:
            logits[:, list(banned)] = -float("inf")
        if vocab_limit is not None:
            logits[:, vocab_limit:] = -float("inf")     # ids the tokenizer cannot decode
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=device)
            sc = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                bad = seen[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if bad:
                    logits[b, list(bad)] = -float("inf")
        nxt = logits.argmax(dim=-1)
        for b, tok in enumerate(nxt.tolist()):
            generated[b].append(tok)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                g = generated[b][-no_repeat_ngram:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
        if t == max_new_tokens - 1:
            break
        if target_words and t >= 120 and t % 16 == 0 and all(
                len(decode(g).split()) > target_words for g in generated):
            break
        cur = T0 + t
        logits = model.step(nxt[:, None], positions[:, cur:cur + 1], caches,
                            key_ok[:, None, None, :cur + 1])[:, -1, :].float()
    return generated


def build_model(config):
    """Contract function."""
    return PrefixLM(config)


USE_GRAD_CHECKPOINTING = True


def _model_self_test():
    torch.manual_seed(0)
    tiny = dict(MODEL_C_CONFIG, vocab_size=101, d_model=64, n_layer=3, n_head=4, n_kv_head=2,
                d_head=16, d_ff=128, max_seq_len=64)
    m = build_model(tiny).to(DEVICE).eval()
    ids = torch.randint(4, 101, (2, 30), device=DEVICE)
    P = torch.tensor([12, 12], device=DEVICE)
    with torch.no_grad():
        full = m(ids, prefix_len=P)
        caches = [KVCache(2, 2, 30, 16, DEVICE, torch.float32) for _ in range(3)]
        pos = torch.arange(30, device=DEVICE)[None].expand(2, -1)
        parts = [m.step(ids[:, :12], pos[:, :12], caches, torch.ones(2, 1, 12, 12, dtype=torch.bool, device=DEVICE))]
        for t in range(12, 30):
            parts.append(m.step(ids[:, t:t + 1], pos[:, t:t + 1], caches,
                                torch.ones(2, 1, 1, t + 1, dtype=torch.bool, device=DEVICE)))
        diff = (full - torch.cat(parts, 1)).abs().max().item()
    assert diff < 1e-3, f"cached generation != full forward ({diff})"

    ids2 = ids.clone()
    ids2[:, 20] = (ids[:, 20] + 1) % 97 + 4          # change a continuation token
    ids3 = ids.clone()
    ids3[:, 5] = (ids[:, 5] + 1) % 97 + 4            # change a prompt token
    with torch.no_grad():
        leak = (m(ids2, prefix_len=P)[:, :20] - full[:, :20]).abs().max().item()
        sees_right = (m(ids3, prefix_len=P)[:, 2] - full[:, 2]).abs().max().item()
        causal_ok = (m(ids3)[:, 2] - m(ids)[:, 2]).abs().max().item()
    assert leak < 1e-5, f"future continuation leaked into earlier positions ({leak})"
    assert sees_right > 1e-5, "prompt tokens do not see later prompt tokens"
    assert causal_ok < 1e-5, "causal mode (prefix 0) sees the future"

    targets = torch.randint(4, 101, (2, 30), device=DEVICE)
    with torch.no_grad():
        s = m(ids, prefix_len=P, targets=targets)[0]
        ref = F.cross_entropy(full[:, 11:].reshape(-1, 101), targets[:, 11:].reshape(-1), reduction="sum")
    assert abs(s[0].item() - ref.item()) < 1e-3 and int(s[1].item()) == 2 * 19, "loss masking wrong"

    pa, pb = ids[0, :20].tolist(), ids[1, :9].tolist()
    both = generate_batch(m, [pa, pb], max_new_tokens=10)
    alone = [generate_batch(m, [pa], max_new_tokens=10)[0], generate_batch(m, [pb], max_new_tokens=10)[0]]
    assert both == alone, "batched (padded) generation differs from one-at-a-time"
    print(f"Self-test passed: cache-vs-full {diff:.1e}, no leak {leak:.1e}, prompt two-way ok, "
          f"loss mask ok, batched generation = single.")


_model_self_test()
torch.manual_seed(1234)
model = build_model(MODEL_C_CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model C parameters (tied embeddings): {n_params / 1e6:.1f}M")
assert 50e6 <= n_params <= 300e6


Self-test passed: cache-vs-full 2.4e-07, no leak 0.0e+00, prompt two-way ok, loss mask ok, batched generation = single.
Model C parameters (tied embeddings): 112.8M


## 5. Two GPUs, batch size, and the LR schedule

Same as Model B: DataParallel self-test with automatic single-GPU fallback;
a memory/speed probe at `seq_len=1024` (weights restored afterwards); the
cosine schedule sized so it fully winds down by `TOTAL_TRAIN_HOURS`.


In [5]:
def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda")
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler()


def reset_gpu_memory():
    gc.collect()
    for d in range(N_GPUS):
        with torch.cuda.device(d):
            torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(d)


def is_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower()


def make_optimizer(module, lr):
    decay = [p for p in module.parameters() if p.dim() >= 2]
    no_decay = [p for p in module.parameters() if p.dim() < 2]      # norms and biases
    return torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                              {"params": no_decay, "weight_decay": 0.0}], lr=lr, betas=(0.9, 0.95), eps=1e-8)


def loss_from(out):
    """out: [n_replicas, 3] sums -> (mean CE, loss incl. z-loss)."""
    s = out.sum(0)
    n = s[1].clamp(min=1)
    return s[0] / n, (s[0] + Z_LOSS_COEF * s[2]) / n


def random_batch(bs, T):
    x = torch.randint(4, MODEL_C_CONFIG["vocab_size"], (bs, T), device=DEVICE)
    y = torch.randint(4, MODEL_C_CONFIG["vocab_size"], (bs, T), device=DEVICE)
    p = torch.tensor([PREFIX_MIN if i % 2 == 0 else 0 for i in range(bs)], device=DEVICE)
    return x, y, p


train_model, USING_DP, GPUS_USED = model, False, 1
if USE_BOTH_GPUS and N_GPUS >= 2:
    try:
        dp_model = nn.DataParallel(model, device_ids=[0, 1])
        model.train()
        x_t, y_t, p_t = random_batch(4, 512)
        p_t = torch.tensor([300, 0, 200, 0], device=DEVICE)
        with torch.no_grad():
            single = loss_from(model(x_t, prefix_len=p_t, targets=y_t))[0]
            multi = loss_from(dp_model(x_t, prefix_len=p_t, targets=y_t))[0]
        gap = abs(single.item() - multi.item())
        loss_from(dp_model(x_t, prefix_len=p_t, targets=y_t))[1].backward()
        grads_ok = all(p.grad is not None and torch.isfinite(p.grad).all().item() for p in model.parameters())
        model.zero_grad(set_to_none=True)
        if gap < 2e-2 and grads_ok:
            train_model, USING_DP, GPUS_USED = dp_model, True, 2
            print(f"DataParallel self-test passed (loss gap {gap:.1e}) -> training on BOTH GPUs.")
        else:
            print(f"DataParallel self-test failed (gap {gap:.1e}, grads ok={grads_ok}) -> 1 GPU.")
    except Exception as exc:
        model.zero_grad(set_to_none=True)
        print(f"DataParallel unavailable ({type(exc).__name__}: {exc}) -> 1 GPU.")
else:
    print(f"Training on 1 GPU (GPUs visible: {N_GPUS}).")

MEMORY_BUDGET_GB = 13.0
CANDIDATE_BATCH_SIZES = [b for b in [2, 4, 6, 8, 12, 16, 24, 32] if b % GPUS_USED == 0]
initial_state = {k: v.detach().clone().cpu() for k, v in model.state_dict().items()}


def try_config(bs, T, n_iters=3):
    reset_gpu_memory()
    probe_optim = None
    try:
        probe_optim = make_optimizer(model, 1e-4)
        probe_scaler = make_grad_scaler()
        x, y, p = random_batch(bs, T)
        timings = []
        for _ in range(n_iters):
            torch.cuda.synchronize()
            t0 = time.time()
            probe_optim.zero_grad(set_to_none=True)
            probe_scaler.scale(loss_from(train_model(x, prefix_len=p, targets=y))[1]).backward()
            probe_scaler.step(probe_optim)
            probe_scaler.update()
            torch.cuda.synchronize()
            timings.append(time.time() - t0)
        return max(torch.cuda.max_memory_allocated(d) for d in range(GPUS_USED)) / 1e9, min(timings[1:])
    except Exception as exc:
        if is_oom(exc):
            return None, None
        raise
    finally:
        del probe_optim
        model.zero_grad(set_to_none=True)
        reset_gpu_memory()


model.train()
probe = {}
for bs in CANDIDATE_BATCH_SIZES:
    if bs * TRAIN_SEQ_LEN > TOKENS_PER_STEP_TARGET:
        break
    peak, sec = try_config(bs, TRAIN_SEQ_LEN)
    if peak is None:
        print(f"  batch={bs:3d} -> OOM")
        break
    fits = peak <= MEMORY_BUDGET_GB
    print(f"  batch={bs:3d} -> {peak:5.2f} GB/GPU, {sec:.2f}s/micro-step, "
          f"{bs * TRAIN_SEQ_LEN / sec:,.0f} tok/s" + ("" if fits else "  [over budget]"))
    if not fits:
        break
    probe[bs] = sec
model.load_state_dict(initial_state)
del initial_state
reset_gpu_memory()
assert probe, "Nothing fit in memory."

MICRO_BATCH_SIZE = max(probe)
SEC_PER_MICRO_STEP = probe[MICRO_BATCH_SIZE]
GRAD_ACCUM_STEPS = 1 if SMOKE_MODE else max(1, round(TOKENS_PER_STEP_TARGET / (MICRO_BATCH_SIZE * TRAIN_SEQ_LEN)))
TOKENS_PER_STEP = MICRO_BATCH_SIZE * TRAIN_SEQ_LEN * GRAD_ACCUM_STEPS
planned_steps = max(50, int(TOTAL_TRAIN_HOURS * 3600 / (SEC_PER_MICRO_STEP * GRAD_ACCUM_STEPS * 1.2)))
SCHEDULE = {
    "peak_lr": PEAK_LR, "min_lr": MIN_LR,
    "warmup_steps": min(200, max(10, planned_steps // 20)),
    "lr_decay_steps": planned_steps,
    "tokens_per_step": TOKENS_PER_STEP,
    "total_train_hours": TOTAL_TRAIN_HOURS,
}
print(f"\nGPUs used: {GPUS_USED} | seq_len={TRAIN_SEQ_LEN} micro-batch={MICRO_BATCH_SIZE} "
      f"grad-accum={GRAD_ACCUM_STEPS} -> {TOKENS_PER_STEP:,} tokens/step")
print(f"~{MICRO_BATCH_SIZE * TRAIN_SEQ_LEN / SEC_PER_MICRO_STEP:,.0f} tokens/s -> ~{planned_steps:,} steps "
      f"(~{planned_steps * TOKENS_PER_STEP / 1e6:,.0f}M tokens) in {TOTAL_TRAIN_HOURS}h")
print("Schedule:", SCHEDULE)


DataParallel self-test passed (loss gap 1.3e-05) -> training on BOTH GPUs.
  batch=  2 ->  2.32 GB/GPU, 0.42s/micro-step, 4,842 tok/s
  batch=  4 ->  2.86 GB/GPU, 0.41s/micro-step, 10,046 tok/s
  batch=  6 ->  3.57 GB/GPU, 0.50s/micro-step, 12,313 tok/s
  batch=  8 ->  4.27 GB/GPU, 0.60s/micro-step, 13,745 tok/s
  batch= 12 ->  5.69 GB/GPU, 0.82s/micro-step, 15,007 tok/s
  batch= 16 ->  7.11 GB/GPU, 1.05s/micro-step, 15,678 tok/s
  batch= 24 ->  9.94 GB/GPU, 1.49s/micro-step, 16,489 tok/s
  batch= 32 -> 12.77 GB/GPU, 1.93s/micro-step, 17,001 tok/s

GPUs used: 2 | seq_len=1024 micro-batch=32 grad-accum=1 -> 32,768 tokens/step
~17,001 tokens/s -> ~13,229 steps (~433M tokens) in 8.5h
Schedule: {'peak_lr': 0.0006, 'min_lr': 5.9999999999999995e-05, 'warmup_steps': 200, 'lr_decay_steps': 13229, 'tokens_per_step': 32768, 'total_train_hours': 8.5}


## 6. Checkpoints and Kaggle push

Same contract as A and B. Own dataset: `legallm-checkpoints-c`.


In [6]:
from kaggle_secrets import UserSecretsClient

KAGGLE_USERNAME = "jatiniitk"
CHECKPOINT_DATASET_SLUG = "legallm-checkpoints-c"


def save_checkpoint(path, model, optim, sched, scaler, step, tokens_seen, config, extra=None):
    state = {
        "model_state": model.state_dict(), "optim_state": optim.state_dict(),
        "scheduler_state": sched.state_dict() if sched is not None else None,
        "scaler_state": scaler.state_dict(), "step": step, "tokens_seen": tokens_seen,
        "rng_state": torch.get_rng_state(), "config": config, "arch_id": ARCH_ID,
    }
    if extra:
        state.update(extra)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(state, tmp)
    tmp.replace(path)


def load_checkpoint(path, map_location):
    return torch.load(path, map_location=map_location, weights_only=False)


def step_of(path):
    return int(Path(path).stem.split("_step")[1].split("_tokens")[0])


def checkpoint_path_for(step, tokens_seen):
    return CHECKPOINT_DIR / f"{CKPT_PREFIX}_step{step}_tokens{tokens_seen}.pt"


def find_latest_checkpoint(prefix):
    if not KAGGLE_INPUT.exists():
        return None
    pts = list(KAGGLE_INPUT.rglob(f"{prefix}_step*.pt"))
    if pts:
        return max(pts, key=step_of)
    best = None
    for zpath in KAGGLE_INPUT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(zpath) as zf:
                for name in zf.namelist():
                    base = Path(name).name
                    if base.startswith(f"{prefix}_step") and base.endswith(".pt"):
                        if best is None or step_of(base) > best[0]:
                            best = (step_of(base), zpath, name)
        except zipfile.BadZipFile:
            continue
    if best is None:
        return None
    _, zpath, name = best
    dest = CHECKPOINT_DIR / Path(name).name
    with zipfile.ZipFile(zpath) as zf, zf.open(name) as src, open(dest, "wb") as dst:
        shutil.copyfileobj(src, dst)
    return dest


def get_kaggle_token():
    try:
        token = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    except Exception as exc:
        print(f"Could not read the 'KAGGLE_API_TOKEN' secret ({exc}).")
        return None
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / "access_token").write_text(token)
    (kaggle_dir / "access_token").chmod(0o600)
    return token


def prune_old_checkpoints(keep=2):
    ckpts = sorted(CHECKPOINT_DIR.glob(f"{CKPT_PREFIX}_step*.pt"), key=step_of)
    for p in ckpts[:-keep] if len(ckpts) > keep else []:
        p.unlink()


def push_checkpoint_dataset(message):
    if get_kaggle_token() is None:
        print("Skipping checkpoint push: no KAGGLE_API_TOKEN secret.")
        return False
    dataset_id = f"{KAGGLE_USERNAME}/{CHECKPOINT_DATASET_SLUG}"
    with open(OUTPUT_DIR / "dataset-metadata.json", "w") as f:
        json.dump({"title": CHECKPOINT_DATASET_SLUG, "id": dataset_id, "licenses": [{"name": "CC0-1.0"}]}, f)
    exists = subprocess.run(["kaggle", "datasets", "status", dataset_id], capture_output=True, text=True).returncode == 0
    cmd = (["kaggle", "datasets", "version", "-p", str(OUTPUT_DIR), "-m", message, "-r", "zip"] if exists
           else ["kaggle", "datasets", "create", "-p", str(OUTPUT_DIR), "-r", "zip"])
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Checkpoint push failed: {(result.stderr or result.stdout).strip()[-500:]}")
        return False
    print(f"Pushed checkpoint dataset in {time.time() - t0:.0f}s ({message}).")
    return True


print(f"Push target: {KAGGLE_USERNAME}/{CHECKPOINT_DATASET_SLUG}")


Push target: jatiniitk/legallm-checkpoints-c


## 7. Optimizer, schedule, and resume

In [7]:
optimizer = make_optimizer(model, PEAK_LR)
scaler = make_grad_scaler()


def get_lr(step):
    s = SCHEDULE
    if step < s["warmup_steps"]:
        return s["peak_lr"] * (step + 1) / s["warmup_steps"]
    progress = min(1.0, (step - s["warmup_steps"]) / max(1, s["lr_decay_steps"] - s["warmup_steps"]))
    return s["min_lr"] + (s["peak_lr"] - s["min_lr"]) * 0.5 * (1 + math.cos(math.pi * progress))


start_step, start_tokens_seen, resume_file_positions = 0, 0, {}
latest = find_latest_checkpoint(CKPT_PREFIX)
if latest is None:
    print(f"No {CKPT_PREFIX}_step*.pt checkpoint attached -- starting fresh.")
else:
    print(f"Resuming from {latest}")
    ckpt = load_checkpoint(latest, map_location="cpu")
    assert ckpt["config"] == MODEL_C_CONFIG, "Checkpoint config does not match MODEL_C_CONFIG!"
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optim_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    torch.set_rng_state(ckpt["rng_state"])
    start_step, start_tokens_seen = ckpt["step"], ckpt["tokens_seen"]
    resume_file_positions = ckpt.get("data_state", {}).get("file_positions", {})
    if ckpt.get("schedule"):
        SCHEDULE = ckpt["schedule"]
    if latest.parent != CHECKPOINT_DIR:
        shutil.copy(latest, CHECKPOINT_DIR / latest.name)
    del ckpt
    gc.collect()
    print(f"Resumed at step={start_step:,}, tokens_seen={start_tokens_seen:,}")
if LR_DECAY_STEPS_OVERRIDE is not None:
    SCHEDULE = dict(SCHEDULE, lr_decay_steps=int(LR_DECAY_STEPS_OVERRIDE))
print(f"LR now {get_lr(start_step):.2e}; decays to {SCHEDULE['min_lr']:.0e} by step {SCHEDULE['lr_decay_steps']:,}")


No C_step*.pt checkpoint attached -- starting fresh.
LR now 3.00e-06; decays to 6e-05 by step 13,229


## 8. Data pipeline

Round-robin over the four cleaned text files, documents separated by
`<eos>`, byte positions saved in each checkpoint (as in B). Each
1024-token window gets a prompt length: 0 (plain next-word training) or,
for `TASK_FRACTION` of windows, a random length between 480 and 832
tokens (shaped like the test).


In [8]:
class PrefixStreamDataset(torch.utils.data.IterableDataset):
    def __init__(self, file_paths, seq_len, start_positions=None, seed=0):
        self.file_paths = [p for p in file_paths if p.stat().st_size > 0]
        self.seq_len = seq_len
        start_positions = start_positions or {}
        self.positions = {p.name: int(start_positions.get(p.name, 0)) for p in self.file_paths}
        self.rng = random.Random(seed)

    def __iter__(self):
        buffer = []
        readers = [self._read_lines(p) for p in self.file_paths]
        for reader in itertools.cycle(readers):
            ids = encode(next(reader))
            if not ids:
                continue
            buffer.extend(ids)
            buffer.append(EOS_ID)
            while len(buffer) >= self.seq_len + 1:
                window, buffer = buffer[:self.seq_len + 1], buffer[self.seq_len + 1:]
                prefix = self.rng.randint(PREFIX_MIN, PREFIX_MAX) if self.rng.random() < TASK_FRACTION else 0
                yield (torch.tensor(window[:-1], dtype=torch.long), torch.tensor(window[1:], dtype=torch.long),
                       torch.tensor(prefix, dtype=torch.long))

    def _read_lines(self, path):
        name, size = path.name, path.stat().st_size
        while True:
            start = self.positions[name] if self.positions[name] < size else 0
            with open(path, "rb") as f:
                f.seek(start)
                pos = start
                while True:
                    raw = f.readline()
                    if not raw:
                        break
                    pos += len(raw)
                    self.positions[name] = pos
                    line = raw.decode("utf-8", errors="ignore").strip()
                    if line:
                        yield line
            self.positions[name] = 0


TRAIN_FILES = sorted(INPUT_DATA_DIR.glob("clean_*.txt"))
assert TRAIN_FILES, "No clean_*.txt files -- check legallm-phase1-data."
train_dataset = PrefixStreamDataset(TRAIN_FILES, TRAIN_SEQ_LEN, resume_file_positions, seed=start_step)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=MICRO_BATCH_SIZE, num_workers=0)

_check = PrefixStreamDataset(TRAIN_FILES, TRAIN_SEQ_LEN, dict(train_dataset.positions), seed=99)
_x, _y, _p = next(iter(torch.utils.data.DataLoader(_check, batch_size=MICRO_BATCH_SIZE)))
assert torch.equal(_x[:, 1:], _y[:, :-1])
print("Training on:", [p.name for p in TRAIN_FILES])
print(f"Sample batch: x={tuple(_x.shape)}, prompt lengths={_p.tolist()}")
del _check, _x, _y, _p


Training on: ['clean_acts.txt', 'clean_high_courts.txt', 'clean_supreme_court.txt', 'clean_tribunals.txt']
Sample batch: x=(32, 1024), prompt lengths=[582, 0, 548, 0, 676, 0, 0, 799, 695, 0, 0, 0, 0, 0, 0, 716, 0, 0, 750, 0, 641, 617, 0, 665, 0, 0, 581, 0, 0, 590, 0, 0]


## 9. Training loop

Same as B: time budget, checkpoint + push every `CHECKPOINT_EVERY_MIN`,
tqdm bar with timestamp, non-finite steps skipped (abort only after 25 in
a row, after saving). The loss shown is averaged over the tokens that are
actually trained on (continuations in task windows, everything in LM windows).


In [9]:
from tqdm.auto import tqdm

run_id = f"{ARCH_ID}_{'smoke' if SMOKE_MODE else 'main'}_{time.strftime('%Y%m%d_%H%M%S')}"
print(f"run_id = {run_id}")


def training_extras():
    return {"schedule": SCHEDULE, "data_state": {"file_positions": dict(train_dataset.positions)},
            "run_mode": "smoke" if SMOKE_MODE else "main"}


def save_now():
    path = checkpoint_path_for(step, tokens_seen)
    save_checkpoint(path, model, optimizer, None, scaler, step, tokens_seen, MODEL_C_CONFIG, extra=training_extras())
    prune_old_checkpoints(keep=2)
    return path


model.train()
data_iter = iter(train_loader)
step, tokens_seen = start_step, start_tokens_seen
t_start = last_ckpt_time = time.time()
bad_streak = skipped_steps = 0
stop_reason = "unknown"
progress_bar = tqdm(total=round(TIME_BUDGET_MIN, 1), unit="min", desc=run_id, dynamic_ncols=True)
last_reported_min = 0.0
try:
    while True:
        elapsed_min = (time.time() - t_start) / 60
        if elapsed_min >= TIME_BUDGET_MIN:
            stop_reason = "time budget reached"
            break
        lr = get_lr(step)
        for group in optimizer.param_groups:
            group["lr"] = lr
        optimizer.zero_grad(set_to_none=True)
        ce_sum = 0.0
        for _ in range(GRAD_ACCUM_STEPS):
            x, y, p = next(data_iter)
            x, y, p = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True), p.to(DEVICE)
            ce, total = loss_from(train_model(x, prefix_len=p, targets=y))
            scaler.scale(total / GRAD_ACCUM_STEPS).backward()
            ce_sum += ce.item()
            tokens_seen += x.numel()
        ce = ce_sum / GRAD_ACCUM_STEPS
        scaler.unscale_(optimizer)
        grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM))
        scaler.step(optimizer)
        scaler.update()
        if math.isfinite(ce) and math.isfinite(grad_norm):
            bad_streak = 0
        else:
            bad_streak += 1
            skipped_steps += 1
            if bad_streak >= 25:
                stop_reason = "25 non-finite steps in a row"
                break
        if step % 20 == 0 or step == start_step:
            print(f"  step {step:6d}  loss={ce:.4f}  lr={lr:.2e}  gnorm={grad_norm:.2f}  "
                  f"tokens_seen={tokens_seen:,}  elapsed={elapsed_min:.1f} min")
        now_min = (time.time() - t_start) / 60
        progress_bar.update(now_min - last_reported_min)
        last_reported_min = now_min
        progress_bar.set_postfix({"time": datetime.datetime.now().strftime("%H:%M:%S"), "step": step,
                                  "loss": f"{ce:.3f}", "lr": f"{lr:.1e}", "tokens": f"{tokens_seen:,}"})
        step += 1
        if (time.time() - last_ckpt_time) / 60 >= CHECKPOINT_EVERY_MIN:
            path = save_now()
            print(f"  [checkpoint saved -> {path.name}]")
            if PUSH_DURING_TRAINING:
                push_checkpoint_dataset(f"{run_id}: step={step}, tokens_seen={tokens_seen:,} (in progress)")
            last_ckpt_time = time.time()
except KeyboardInterrupt:
    stop_reason = "stopped by user"
finally:
    progress_bar.close()

final_path = save_now()
total_min = (time.time() - t_start) / 60
print(f"\nStopped ({stop_reason}) at step={step:,}, tokens_seen={tokens_seen:,}, after {total_min:.1f} min "
      f"(finished {datetime.datetime.now():%Y-%m-%d %H:%M:%S}). Skipped steps: {skipped_steps}.")
_re = load_checkpoint(final_path, map_location="cpu")
assert all(torch.equal(_re["model_state"][k], v.detach().cpu()) for k, v in model.state_dict().items())
print(f"Final checkpoint -> {final_path.name} (round-trip check passed)")
del _re
gc.collect()
if stop_reason == "25 non-finite steps in a row":
    raise RuntimeError("Training diverged (checkpoint saved). Lower PEAK_LR and resume.")


run_id = C_main_20260930_054318


C_main_20260930_054318:   0%|          | 0/510 [00:00<?, ?min/s]

  step      0  loss=10.4658  lr=3.00e-06  gnorm=24.83  tokens_seen=32,768  elapsed=0.0 min
  step     20  loss=8.0832  lr=6.30e-05  gnorm=2.93  tokens_seen=688,128  elapsed=0.7 min
  step     40  loss=6.8304  lr=1.23e-04  gnorm=1.79  tokens_seen=1,343,488  elapsed=1.4 min
  step     60  loss=6.0660  lr=1.83e-04  gnorm=1.51  tokens_seen=1,998,848  elapsed=2.1 min
  step     80  loss=5.5757  lr=2.43e-04  gnorm=1.64  tokens_seen=2,654,208  elapsed=2.9 min
  step    100  loss=5.7054  lr=3.03e-04  gnorm=1.77  tokens_seen=3,309,568  elapsed=3.6 min
  step    120  loss=5.5066  lr=3.63e-04  gnorm=1.16  tokens_seen=3,964,928  elapsed=4.3 min
  step    140  loss=5.3436  lr=4.23e-04  gnorm=1.91  tokens_seen=4,620,288  elapsed=5.1 min
  step    160  loss=5.4360  lr=4.83e-04  gnorm=0.87  tokens_seen=5,275,648  elapsed=5.8 min
  step    180  loss=5.1196  lr=5.43e-04  gnorm=1.53  tokens_seen=5,931,008  elapsed=6.5 min
  step    200  loss=4.9171  lr=6.00e-04  gnorm=0.99  tokens_seen=6,586,368  elapsed

In [10]:
from IPython.display import FileLink, display

display(FileLink(str(final_path.relative_to(Path.cwd())) if final_path.is_relative_to(Path.cwd()) else str(final_path)))
print(f"({final_path.stat().st_size / 1e6:.0f} MB, model + optimizer state)")


/kaggle/working/phase1_model_c/C_step13683_tokens448364544.pt

(1353 MB, model + optimizer state)


## 10. Dev loss and bits-per-byte

The whole dev prompt is the prefix (seen in both directions), and only the
reference continuation is scored -- exactly the test setting.
Bits-per-byte is comparable with B's number despite the different tokenizer.


In [11]:
with open(INPUT_DATA_DIR / "dev_prompts.json") as f:
    dev_prompts = json.load(f)
with open(INPUT_DATA_DIR / "dev_truth.json") as f:
    dev_truth = json.load(f)
dev_ids = sorted(set(dev_prompts) & set(dev_truth))

model.eval()
total_nll, total_tokens, total_bytes = 0.0, 0, 0
with torch.no_grad():
    for doc_id in dev_ids[:200]:
        cont_text = " " + dev_truth[doc_id]["text"].strip()
        cont_ids = encode(cont_text)
        room = TRAIN_SEQ_LEN - len(cont_ids)
        if room < 16 or not cont_ids:
            continue
        ctx = encode(dev_prompts[doc_id]["text"])[-room:]
        ids_t = torch.tensor([ctx + cont_ids], device=DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(ids_t, prefix_len=torch.tensor([len(ctx)], device=DEVICE))
        n = len(ctx)
        total_nll += F.cross_entropy(logits[0, n - 1:-1].float(), ids_t[0, n:], reduction="sum").item()
        total_tokens += len(cont_ids)
        total_bytes += len(cont_text.encode("utf-8"))
model.train()
val_loss = total_nll / total_tokens if total_tokens else float("nan")
bits_per_byte = total_nll / (math.log(2) * total_bytes) if total_bytes else float("nan")
print(f"Dev continuation loss: {val_loss:.4f}/token   bits-per-byte: {bits_per_byte:.4f}  (compare with B's)")


Dev continuation loss: 2.5243/token   bits-per-byte: 0.7533  (compare with B's)


## 11. Generate 100 dev predictions and score them

Uses the output setting that won for both A and B: repetition penalty 1.2,
no repeated 3-token phrase, 128 words. (The separate prediction notebook
re-checks a few settings before making the test submission.)


In [12]:
GEN = dict(max_new_tokens=288, max_context=TRAIN_SEQ_LEN, rep_penalty=1.2, no_repeat_ngram=3,
           target_words=128, banned=BANNED_IN_OUTPUT, vocab_limit=sp.get_piece_size())
sample_ids = dev_ids[:10] if SMOKE_MODE else dev_ids[:100]
predictions = {}
t0 = time.time()
for s in tqdm(range(0, len(sample_ids), 16), desc="generating", dynamic_ncols=True):
    chunk = sample_ids[s:s + 16]
    outs = generate_batch(model, [encode(dev_prompts[i]["text"]) for i in chunk], **GEN)
    for i, toks in zip(chunk, outs):
        words = decode(toks).split()[:128]
        predictions[i] = {"text": " ".join(words) if words else "."}
model.train()
print(f"Generated {len(predictions)} in {time.time() - t0:.0f}s.")
print("Example:", predictions[sample_ids[0]]["text"][:300])
predictions_path = OUTPUT_DIR / "dev_predictions.json"
json.dump(predictions, open(predictions_path, "w"), indent=1)

try:
    import bert_score
except ImportError:
    subprocess.run(["pip", "install", "-q", "bert-score"], check=True)
    import bert_score


def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return state.bit_count() if hasattr(int, "bit_count") else bin(state).count("1")


def rouge_l_f1(p_text, r_text):
    pt, rt = p_text.split(), r_text.split()
    if not pt or not rt:
        return 0.0
    l = lcs_length(pt, rt)
    if not l:
        return 0.0
    pr, rc = l / len(pt), l / len(rt)
    return 2 * pr * rc / (pr + rc)


def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)) if len(tokens) >= n else Counter()


def bleu4(preds, refs):
    clipped, totals, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        pl += len(p)
        rl += len(r)
        for n in range(1, 5):
            pn, rn = ngrams(p, n), ngrams(r, n)
            totals[n - 1] += sum(pn.values())
            for g, c in pn.items():
                clipped[n - 1] += min(c, rn.get(g, 0))
    prec = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x <= 0 for x in prec) or pl == 0:
        return 0.0
    return (1.0 if pl > rl else math.exp(1.0 - rl / pl)) * math.exp(sum(math.log(x) for x in prec) / 4)


refs = [dev_truth[i]["text"].strip() for i in sample_ids]
hyps = [predictions[i]["text"].strip() for i in sample_ids]
_, _, f1 = bert_score.score(hyps, refs, lang="en", batch_size=8, verbose=False,
                            device="cuda" if torch.cuda.is_available() else "cpu")
dev_bertscore = float(f1.mean())
dev_rouge_l = sum(rouge_l_f1(h, r) for h, r in zip(hyps, refs)) / len(refs)
dev_bleu4 = bleu4([h.split() for h in hyps], [r.split() for r in refs])
print(f"Examples: {len(refs)}\nBERTScore F1: {dev_bertscore:.6f}\nROUGE-L F1:   {dev_rouge_l:.6f}\nBLEU-4:       {dev_bleu4:.6f}")
print("Compare (same 100 IDs, same output setting): A 0.8245, B 0.8393.")


generating:   0%|          | 0/7 [00:00<?, ?it/s]

Generated 100 in 46s.
Example: interest in land and the aggregate of such interest in any building; (9) "land revenue" means the amount of land revenue payable by a village officer under this Act; (10) "land-revenue" includes the additional tax, if any, levied on the land or buildings; (11) "lands and buildings" means any structu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.5 MB/s eta 0:00:00


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Examples: 100
BERTScore F1: 0.843198
ROUGE-L F1:   0.230000
BLEU-4:       0.113453
Compare (same 100 IDs, same output setting): A 0.8245, B 0.8393.


## 12. `runs.csv` and final publish

In [13]:
import csv

RUNS_CSV_PATH = OUTPUT_DIR / "runs.csv"
RUNS_CSV_COLUMNS = ["run_id", "arch_id", "technique", "params", "tokens_seen", "tokens_per_sec", "peak_mem_gb",
                    "val_loss", "bits_per_byte", "dev_bertscore", "dev_rouge_l", "dev_bleu4", "codabench_score",
                    "checkpoint_path", "notes"]
session_tokens = tokens_seen - start_tokens_seen
row = {
    "run_id": run_id, "arch_id": ARCH_ID, "technique": "full-pretrain (prefix-LM)", "params": n_params,
    "tokens_seen": tokens_seen,
    "tokens_per_sec": round(session_tokens / (total_min * 60), 1) if total_min > 0 else None,
    "peak_mem_gb": round(max(torch.cuda.max_memory_allocated(d) for d in range(GPUS_USED)) / 1e9, 2) if N_GPUS else 0,
    "val_loss": round(val_loss, 4), "bits_per_byte": round(bits_per_byte, 4),
    "dev_bertscore": round(dev_bertscore, 4), "dev_rouge_l": round(dev_rouge_l, 4), "dev_bleu4": round(dev_bleu4, 4),
    "codabench_score": None, "checkpoint_path": final_path.name,
    "notes": (f"{'smoke' if SMOKE_MODE else 'main'}; {GPUS_USED} GPU(s); seq_len={TRAIN_SEQ_LEN}; "
              f"task_fraction={TASK_FRACTION}; T2; lr_decay_steps={SCHEDULE['lr_decay_steps']}; "
              f"gen=rep1.2+norepeat3->128w; stop={stop_reason}"),
}
prev = [p for p in KAGGLE_INPUT.rglob("runs.csv") if CHECKPOINT_DATASET_SLUG in str(p)] if KAGGLE_INPUT.exists() else []
rows = list(csv.DictReader(open(prev[0]))) if prev else []
rows = [r for r in rows if r.get("run_id") != run_id] + [row]
with open(RUNS_CSV_PATH, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=RUNS_CSV_COLUMNS, extrasaction="ignore")
    w.writeheader()
    w.writerows(rows)
print(json.dumps(row, indent=2))

prune_old_checkpoints(keep=2)
if push_checkpoint_dataset(f"{run_id} final: step={step}, dev_bertscore={dev_bertscore:.4f}"):
    print(f"Done. Model C is in '{CHECKPOINT_DATASET_SLUG}' (and in this notebook's Output tab).")
else:
    print("Push failed -- the checkpoint is still in this notebook's Output tab.")


{
  "run_id": "C_main_20260930_054318",
  "arch_id": "C",
  "technique": "full-pretrain (prefix-LM)",
  "params": 112762880,
  "tokens_seen": 448364544,
  "tokens_per_sec": 14651.3,
  "peak_mem_gb": 12.77,
  "val_loss": 2.5243,
  "bits_per_byte": 0.7533,
  "dev_bertscore": 0.8432,
  "dev_rouge_l": 0.23,
  "dev_bleu4": 0.1135,
  "codabench_score": null,
  "checkpoint_path": "C_step13683_tokens448364544.pt",
  "notes": "main; 2 GPU(s); seq_len=1024; task_fraction=0.5; T2; lr_decay_steps=13229; gen=rep1.2+norepeat3->128w; stop=time budget reached"
}
Pushed checkpoint dataset in 44s (C_main_20260930_054318 final: step=13683, dev_bertscore=0.8432).
Done. Model C is in 'legallm-checkpoints-c' (and in this notebook's Output tab).


## 13. Next

Run **p1_08_predict_C.ipynb** (about 15 min) to compare output settings on
the dev examples and build `submission.zip` from this checkpoint.
